In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path('.').resolve().parent / 'src'))
from narrativa import estilo, capitulo, mapa, nota, descartes, kpis, frase
estilo()
capitulo(2, 'De dos archivos que no encajan a un solo catálogo',
         'Películas y series llegaron en archivos separados, con IDs que chocan y columnas que no sirven. Aquí se convierten en una tabla sobre la que se puede decidir.',
         ceja='Notebook 02 de 04 · Limpieza e integración')

In [2]:
mapa(['Auditoría de datos', 'Limpieza e integración', 'Análisis exploratorio', 'Informe narrativo'], 2)

**Cómo leer este notebook.** No contiene lógica de transformación: importa las funciones de `src/limpieza.py` y las ejecuta paso a paso, mostrando el efecto de cada una. La lógica vive en el módulo para que el pipeline sea reproducible; el notebook es la narración de lo que el módulo hace.

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
from limpieza import (
    cargar_datos, prefijar_ids, eliminar_columnas_inservibles, agregar_tipo,
    unificar, explotar, agregar_metricas, construir_catalogo, exportar,
    COLUMNAS_ELIMINADAS, UMBRAL_VOTOS,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

peliculas, series = cargar_datos()
print('películas:', peliculas.shape)
print('series:   ', series.shape)

películas: (16000, 18)
series:    (16000, 16)


## Paso 1 — Prefijo de IDs (`MOV_` / `TV_`)

Los 397 `show_id` que colisionan entre archivos apuntan a títulos distintos. Sin prefijo, la concatenación los fusiona.

In [4]:
colisiones = set(peliculas['show_id']) & set(series['show_id'])
ejemplo = list(colisiones)[0]
print('show_id en conflicto:', ejemplo)
print('  en películas →', peliculas.loc[peliculas.show_id == ejemplo, 'title'].iloc[0])
print('  en series    →', series.loc[series.show_id == ejemplo, 'title'].iloc[0])

p = prefijar_ids(peliculas, 'MOV_')
s = prefijar_ids(series, 'TV_')
print('\ntras prefijar, colisiones restantes:', len(set(p['show_id']) & set(s['show_id'])))

show_id en conflicto: 63498
  en películas → Chico & Rita
  en series    → Close Up with The Hollywood Reporter

tras prefijar, colisiones restantes: 0


## Paso 2 — Eliminar `rating` y `duration`

Las dos columnas se eliminan porque no aportan información, no por comodidad. El motivo de cada una está documentado en el propio módulo.

In [5]:
for columna, motivo in COLUMNAS_ELIMINADAS.items():
    print(f'{columna:10s} → {motivo}')

p = eliminar_columnas_inservibles(p)
s = eliminar_columnas_inservibles(s)
print('\ncolumnas tras la limpieza:')
print(' películas:', list(p.columns))
print(' series:   ', list(s.columns))

rating     → copia exacta de vote_average (100% de coincidencia)
duration   → 100% nula en películas y constante en series

columnas tras la limpieza:
 películas: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
 series:    ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average']


## Paso 3 — Columna `tipo` y concatenación

`type` ya existía con valores `Movie` / `TV Show`: no se crea información nueva, se normaliza al idioma de la audiencia. Los 9 duplicados internos de series sobreviven al prefijo (mismo id dentro del mismo archivo) y se resuelven al concatenar.

In [6]:
catalogo = unificar(peliculas, series)

print('filas esperadas sin deduplicar:', len(peliculas) + len(series))
print('filas del catálogo unificado: ', len(catalogo))
print('diferencia (duplicados internos de series):',
      len(peliculas) + len(series) - len(catalogo))
print()
print(catalogo['tipo'].value_counts().to_string())

filas esperadas sin deduplicar: 32000
filas del catálogo unificado:  31991
diferencia (duplicados internos de series): 9

tipo
Película    16000
Serie       15991


## Paso 4 — Métricas derivadas

**`roi = revenue / budget`**, solo donde ambos son positivos. Es un multiplicador: 1.0 es punto de equilibrio. Queda nulo en las series (no traen datos financieros) y en el 78% de las películas.

**`score_ponderado`**, fórmula tipo IMDb, para que un 10.0 con un voto no encabece ningún ranking.

In [7]:
catalogo = agregar_metricas(catalogo)

print('umbral de votos por tipo:', UMBRAL_VOTOS)
print('títulos con votación suficiente:')
print(catalogo.groupby('tipo')['votos_suficientes'].agg(['sum', 'size']).to_string())
print('\npelículas con ROI calculable:', int(catalogo['roi'].notna().sum()))

umbral de votos por tipo: {'Película': 30, 'Serie': 5}
títulos con votación suficiente:
            sum   size
tipo                  
Película  13217  16000
Serie      7763  15991

películas con ROI calculable: 3540


### Por qué el `score_ponderado` y no la nota cruda

La comparación de abajo es la justificación del indicador en una sola celda: el ranking por `vote_average` está tomado por títulos con uno o dos votos.

In [8]:
cols = ['title', 'tipo', 'vote_count', 'vote_average', 'score_ponderado']

print('TOP 5 por nota cruda (vote_average)')
print(catalogo.nlargest(5, 'vote_average')[cols].to_string(index=False))

print('\nTOP 5 por score ponderado')
print(catalogo.nlargest(5, 'score_ponderado')[cols].to_string(index=False))

TOP 5 por nota cruda (vote_average)
                                           title     tipo  vote_count  vote_average  score_ponderado
               Inácio Garapa, Um Matuto Sonhador Película           1          10.0         6.485183
                                The Photographer Película           1          10.0         6.485183
An Unholy Affair: A Younger Man and a Busty Wife Película           1          10.0         6.485183
                                              It Película           2          10.0         6.595021
                            Swapping Guest House Película           1          10.0         6.485183

TOP 5 por score ponderado
                       title  tipo  vote_count  vote_average  score_ponderado
Wizards Beyond Waverly Place Serie          68          9.30         9.155811
         All the Queen's Men Serie          46          9.30         9.093611
               The Territory Serie          39          9.30         9.060777
      When I Fly To

## Paso 5 — Tablas largas: género, país y reparto

El 26% de las películas lista más de un país y casi todos los títulos listan varios géneros. Agregar sobre la columna cruda contaría `"United States of America, France"` como una categoría propia.

Las tablas largas se guardan **aparte**: la tabla principal conserva una fila por título.

In [9]:
genero_largo = explotar(catalogo, 'genres', 'genero')

print('una fila por título-género:')
print(genero_largo.head(6).to_string(index=False))
print('\nfilas:', len(genero_largo), '| géneros distintos:', genero_largo['genero'].nunique())

una fila por título-género:
  show_id     tipo    genero
MOV_10192 Película    Comedy
MOV_10192 Película Adventure
MOV_10192 Película   Fantasy
MOV_10192 Película Animation
MOV_10192 Película    Family
MOV_27205 Película    Action



filas: 65797 | géneros distintos: 27


## Paso 6 — Pipeline completo y exportación

`construir_catalogo()` encadena todo lo anterior en una llamada. El notebook se puede correr de arriba a abajo sin intervención manual.

In [10]:
catalogo, largos = construir_catalogo()
rutas = exportar(catalogo, largos)

print(f'catálogo unificado: {catalogo.shape[0]:,} filas × {catalogo.shape[1]} columnas\n')
for nombre, tabla in largos.items():
    print(f'  {nombre:8s} {len(tabla):>7,} filas | {tabla.iloc[:, -1].nunique():>5,} valores distintos')
print()
for nombre, ruta in rutas.items():
    print(f'  → {ruta.name}')

catálogo unificado: 31,991 filas × 20 columnas

  genero    65,797 filas |    27 valores distintos
  pais      37,628 filas |   147 valores distintos
  actor    140,404 filas | 59,710 valores distintos

  → catalogo_unificado.csv
  → catalogo_genero.csv
  → catalogo_pais.csv
  → catalogo_actor.csv


In [11]:
catalogo.head()

,show_id,title,director,cast,country,date_added,release_year,genres,language,description,popularity,vote_count,vote_average,budget,revenue,tipo,roi,umbral_votos,votos_suficientes,score_ponderado
0,MOV_10192,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0,Película,4.561217,30,True,6.379952
1,MOV_27205,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0,Película,5.243941,30,True,8.367384
2,MOV_12444,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0,Película,3.817223,30,True,7.741867
3,MOV_38757,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0,Película,2.278699,30,True,7.596832
4,MOV_10191,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0,Película,2.999270,30,True,7.796767


In [12]:
kpis([
    (f"{len(peliculas) + len(series):,}".replace(',', '.'), 'filas en los archivos originales', 'películas y series por separado'),
    (f"{len(set(peliculas['show_id']) & set(series['show_id'])):,}".replace(',', '.'), 'IDs en conflicto resueltos', 'con el prefijo MOV_ / TV_'),
    (f"{len(peliculas) + len(series) - len(catalogo):,}".replace(',', '.'), 'duplicados eliminados', 'repetidos dentro de series'),
    (f"{len(catalogo):,}".replace(',', '.'), 'títulos en el catálogo', 'una fila por título, lista para analizar'),
])

---
## Resultado

`data/processed/catalogo_unificado.csv` — 31.991 títulos, una fila por título, listo para el análisis exploratorio (notebook 03) y para el dashboard.

Los tres archivos largos (`catalogo_genero.csv`, `catalogo_pais.csv`, `catalogo_actor.csv`) se usan para agregar por esas dimensiones sin inflar el conteo de títulos.

In [13]:
nota('El catálogo quedó en una sola tabla, sin IDs mezclados, con una nota que no premia a los títulos de un solo voto y con el retorno '
     'calculado solo donde hay datos para hacerlo. Siguiente paso: **notebook 03**, donde esa tabla se convierte en respuestas.',
     'Lo que deja este paso', 'verde')